<a href="https://colab.research.google.com/github/Bashar-04/CardioChest-AI/blob/main/Text_Gen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -U spacy -q

In [ ]:
import requests

In [ ]:
nlp = spacy.load("en_core_web_sm")

In [ ]:
# =========================
# Arabic Text Generation using LSTM
# =========================

import re
import numpy as np

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Embedding
from tensorflow.keras.utils import to_categorical

# =========================
# 1. Arabic document
# =========================

arabic_text = """
الذكاء الاصطناعي أصبح من أهم المجالات في العصر الحديث.
تستخدم تقنيات الذكاء الاصطناعي في الطب والتعليم والصناعة وتحليل البيانات.
يساعد الذكاء الاصطناعي في بناء أنظمة قادرة على التعلم من البيانات واتخاذ قرارات ذكية.
تعلم الآلة هو أحد فروع الذكاء الاصطناعي ويعتمد على تدريب النماذج باستخدام البيانات.
في مجال التعليم يمكن استخدام الذكاء الاصطناعي لتحليل أداء الطلاب وتقديم محتوى مناسب لكل طالب.
كما يمكن استخدامه في أنظمة الحضور الذكية والتعرف على الوجوه ومعالجة الصور.
تعتمد نماذج التعلم العميق على الشبكات العصبية التي تتكون من طبقات متعددة.
كلما زادت كمية البيانات وتحسنت جودة التدريب أصبح النموذج أكثر قدرة على التنبؤ.
توليد النصوص هو تطبيق مهم من تطبيقات التعلم العميق ومعالجة اللغة الطبيعية.
يقوم النموذج بتعلم تسلسل الكلمات ثم يحاول توقع الكلمة التالية بناء على الكلمات السابقة.
"""

# نكرر النص حتى يصبح عندنا بيانات أكثر للتدريب
all_text = arabic_text * 50

# =========================
# 2. Arabic cleaning and tokenization
# =========================

def clean_arabic_text(text):
    # إزالة التشكيل
    text = re.sub(r'[\u064B-\u065F]', '', text)

    # إزالة الرموز والإبقاء على الحروف العربية والمسافات
    text = re.sub(r'[^\u0600-\u06FF\s]', ' ', text)

    # إزالة المسافات الزائدة
    text = re.sub(r'\s+', ' ', text).strip()

    return text

def separate_punc_arabic(doc_text):
    cleaned_text = clean_arabic_text(doc_text)
    tokens = cleaned_text.split()
    return tokens

tokens = separate_punc_arabic(all_text)

print("Total tokens:", len(tokens))
print("First 20 tokens:", tokens[:20])

# =========================
# 3. Organize tokens into sequences
# =========================

train_len = 10 + 1   # 10 input words + 1 target word
text_sequences = []

for i in range(train_len, len(tokens)):
    seq = tokens[i - train_len:i]
    text_sequences.append(" ".join(seq))

print("Total sequences:", len(text_sequences))
print("Sample sequence:", text_sequences[0])

# =========================
# 4. Tokenization
# =========================

tokenizer = Tokenizer(filters='', lower=False)
tokenizer.fit_on_texts(text_sequences)
sequences = tokenizer.texts_to_sequences(text_sequences)

vocabulary_size = len(tokenizer.word_counts) + 1
print("Vocabulary size:", vocabulary_size)

lengths = set(len(seq) for seq in sequences)
print("Unique sequence lengths:", lengths)

sequences = np.array(sequences)

print("Sequences shape:", sequences.shape)
print("First encoded sequence:", sequences[0])

# =========================
# 5. Prepare X and y
# =========================

X = sequences[:, :-1]
y = sequences[:, -1]

y = to_categorical(y, num_classes=vocabulary_size)

print("X shape:", X.shape)
print("y shape:", y.shape)

# =========================
# 6. Create model
# =========================

def create_model(vocabulary_size, seq_len):
    model = Sequential()

    model.add(Embedding(
        input_dim=vocabulary_size,
        output_dim=50,
        input_length=seq_len
    ))

    model.add(LSTM(100, return_sequences=True))
    model.add(LSTM(100))

    model.add(Dense(100, activation='relu'))
    model.add(Dense(vocabulary_size, activation='softmax'))

    model.compile(
        loss='categorical_crossentropy',
        optimizer='adam',
        metrics=['accuracy']
    )

    return model

model = create_model(vocabulary_size, X.shape[1])
model.summary()

# =========================
# 7. Train model
# =========================

model.fit(X, y, batch_size=64, epochs=50, verbose=1)

# =========================
# 8. Text generation function
# =========================

def generate_text(model, tokenizer, seq_len, seed_text, num_gen_words):
    output_text = []
    input_text = seed_text

    for _ in range(num_gen_words):
        cleaned_input = clean_arabic_text(input_text)

        encoded_text = tokenizer.texts_to_sequences([cleaned_input])[0]
        pad_encoded = pad_sequences(
            [encoded_text],
            maxlen=seq_len,
            truncating='pre'
        )

        prediction = model.predict(pad_encoded, verbose=0)[0]
        pred_word_ind = np.argmax(prediction)

        pred_word = tokenizer.index_word.get(pred_word_ind, "")

        if pred_word == "":
            break

        input_text += " " + pred_word
        output_text.append(pred_word)

    return " ".join(output_text)

# =========================
# 9. Test generation
# =========================

seed_text = "الذكاء الاصطناعي"
generated = generate_text(model, tokenizer, X.shape[1], seed_text, 20)

print("\nSeed text:", seed_text)
print("==================================================================")
print("Generated text:", seed_text + " " + generated)
print("==================================================================")

seed_text = "تعلم الآلة"
generated = generate_text(model, tokenizer, X.shape[1], seed_text, 20)

print("\nSeed text:", seed_text)
print("==================================================================")
print("Generated text:", seed_text + " " + generated)
print("==================================================================")

seed_text = "في مجال التعليم"
generated = generate_text(model, tokenizer, X.shape[1], seed_text, 20)

print("\nSeed text:", seed_text)
print("==================================================================")
print("Generated text:", seed_text + " " + generated)
print("==================================================================")

# =========================
# 10. Save model
# =========================

model.save("arabic_lstm_text_generator.h5")
print("Model saved as arabic_lstm_text_generator.h5")